# Compare solar elevation: NSRDB vs CYL_GHI

Load one station from the merged NSRDB files and the matching CYL_GHI CSV, then plot solar elevation over the configured interval. NSRDB elevation is read from `solar_elev` or derived as `90 - Solar Zenith Angle`; CYL_GHI uses `sun_elev`. The NSRDB series is shifted two records (one hour) earlier for plotting; its source data is not changed.

Edit the paths, station code, and date range in the configuration cell below. The default paths follow `NoteBooks/data_preprocess.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# In Colab, mount Google Drive and use its MyDrive folder.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive')
except ImportError:
    # For local Jupyter, change this to the folder containing NSRDB/ and CYL_GHI/.
    BASE_DIR = Path.home() / 'MyDrive'
NSRDB_MERGED_DIR = BASE_DIR / 'NSRDB' / 'merged_files2'
CYL_GHI_DIR = BASE_DIR / 'CYL_GHI'

STATION_CODE = 'YOUR_STATION_CODE'  # e.g. the matching station_code in CYL_geo/CyL_geo.csv
START = '2017-06-01 00:00'
END = '2017-06-07 23:59'
NSRDB_SHIFT_STEPS = -2  # 30-minute samples: -2 means display one hour earlier
NSRDB_ELEVATION_FIELD = 'solar_elev'
CYL_ELEVATION_FIELD = 'sun_elev'
NSRDB_FIELD_IS_ZENITH_ANGLE = False  # Set True if NSRDB_ELEVATION_FIELD names a zenith-angle column

NSRDB_FILE = NSRDB_MERGED_DIR / f'{STATION_CODE}_merged.csv'
# Resolve the raw CYL_GHI CSV by station-code prefix (same convention as data_preprocess.ipynb).
CYL_MATCHES = sorted(CYL_GHI_DIR.glob(f'{STATION_CODE}*.csv'))
if len(CYL_MATCHES) != 1:
    visible_csvs = sorted(p.name for p in CYL_GHI_DIR.glob('*.csv')) if CYL_GHI_DIR.exists() else []
    raise FileNotFoundError(
        f'Expected one CYL_GHI CSV starting with {STATION_CODE!r} in {CYL_GHI_DIR}; found: {CYL_MATCHES}. '
        f'Visible CSVs: {visible_csvs[:20]}'
    )
CYL_FILE = CYL_MATCHES[0]
print('NSRDB:', NSRDB_FILE)
print('CYL_GHI:', CYL_FILE)

In [ ]:
def load_timestamp(df, label):
    if 'timestamp' in df.columns:
        ts = pd.to_datetime(df['timestamp'], errors='coerce')
    else:
        date_cols = ['Year', 'Month', 'Day', 'Hour', 'Minute']
        missing = [c for c in date_cols if c not in df.columns]
        if missing:
            raise KeyError(f'{label}: no timestamp column; missing date fields: {missing}')
        ts = pd.to_datetime(df[date_cols].rename(columns={
            'Year': 'year', 'Month': 'month', 'Day': 'day',
            'Hour': 'hour', 'Minute': 'minute'
        }), errors='coerce')
    return ts

if not NSRDB_FILE.exists():
    raise FileNotFoundError(f'NSRDB merged file not found: {NSRDB_FILE}')
if not CYL_FILE.exists():
    raise FileNotFoundError(f'CYL_GHI file not found: {CYL_FILE}')

nsrdb = pd.read_csv(NSRDB_FILE)
cyl = pd.read_csv(CYL_FILE)

if NSRDB_ELEVATION_FIELD not in nsrdb.columns:
    raise KeyError(f'NSRDB field {NSRDB_ELEVATION_FIELD!r} not found in {NSRDB_FILE}; available fields: {list(nsrdb.columns)}')
nsrdb_elev = pd.to_numeric(nsrdb[NSRDB_ELEVATION_FIELD], errors='coerce')
if NSRDB_FIELD_IS_ZENITH_ANGLE:
    nsrdb_elev = 90 - nsrdb_elev

if CYL_ELEVATION_FIELD not in cyl.columns:
    raise KeyError(f'CYL_GHI field {CYL_ELEVATION_FIELD!r} not found in {CYL_FILE}; available fields: {list(cyl.columns)}')

nsrdb_plot = pd.DataFrame({'timestamp': load_timestamp(nsrdb, 'NSRDB'), 'NSRDB solar_elevation': nsrdb_elev})
cyl_plot = pd.DataFrame({'timestamp': load_timestamp(cyl, 'CYL_GHI'), 'CYL_GHI solar_elevation': pd.to_numeric(cyl[CYL_ELEVATION_FIELD], errors='coerce')})
nsrdb_plot = nsrdb_plot.dropna(subset=['timestamp']).drop_duplicates('timestamp').set_index('timestamp').sort_index()
cyl_plot = cyl_plot.dropna(subset=['timestamp']).drop_duplicates('timestamp').set_index('timestamp').sort_index()

# Move NSRDB values two sample intervals earlier on the time axis (without changing values).
nsrdb_plot.index = nsrdb_plot.index + pd.to_timedelta(NSRDB_SHIFT_STEPS * 30, unit='min')

start, end = pd.Timestamp(START), pd.Timestamp(END)
nsrdb_plot = nsrdb_plot.loc[start:end]
cyl_plot = cyl_plot.loc[start:end]
comparison = nsrdb_plot.join(cyl_plot, how='outer').sort_index()
print(f'NSRDB rows in range: {len(nsrdb_plot):,}; CYL_GHI rows in range: {len(cyl_plot):,}')
print(f'Overlapping timestamps: {nsrdb_plot.index.intersection(cyl_plot.index).size:,}')
comparison.head()

In [ ]:
ax = comparison.plot(figsize=(14, 5), marker='.', markersize=3, linewidth=1)
ax.set_title(f'Solar elevation comparison — {STATION_CODE} ({START} to {END})')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Solar elevation (degrees)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()